# Imports and Load Cleaned Data

In [1]:
# Cell 1
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import json
import os

df = pd.read_csv('../data/cleaned_churn.csv')
print(f"Cleaned data loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

Cleaned data loaded successfully!
Shape: 7,032 rows × 21 columns
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


# Drop Irrelevant Column

In [2]:
# Drop customerID
# It's just a unique identifier — knowing someone's ID number tells
# the model nothing about whether they'll churn

before_cols = len(df.columns)
df = df.drop('customerID', axis=1)

print(f"Dropped 'customerID' column.")
print(f"Columns before: {before_cols}  →  Columns after: {len(df.columns)}")
print(f"Remaining columns: {list(df.columns)}")

Dropped 'customerID' column.
Columns before: 21  →  Columns after: 20
Remaining columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


# Encode Binary Columns

In [3]:
# Convert Yes/No text columns into 1/0 numbers
# ML models only understand numbers — not words
# ANALOGY: Translating a questionnaire from English into a 0-or-1 scale

binary_map  = {'Yes': 1, 'No': 0}
service_map = {'Yes': 1, 'No': 0,
               'No phone service': 0, 'No internet service': 0}

simple_binary = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
service_cols  = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in simple_binary:
    df[col] = df[col].map(binary_map)

for col in service_cols:
    df[col] = df[col].map(service_map)

df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

print(f"Binary encoding complete.")
print(f"\nVerification — NaN values after encoding (all should be 0):")
for col in simple_binary + service_cols + ['gender']:
    nan_count = df[col].isnull().sum()
    status    = 'OK' if nan_count == 0 else f'WARNING: {nan_count} NaN values!'
    print(f"  {col:<30} {status}")

Binary encoding complete.

Verification — NaN values after encoding (all should be 0):
  Partner                        OK
  Dependents                     OK
  PhoneService                   OK
  PaperlessBilling               OK
  Churn                          OK
  MultipleLines                  OK
  OnlineSecurity                 OK
  OnlineBackup                   OK
  DeviceProtection               OK
  TechSupport                    OK
  StreamingTV                    OK
  StreamingMovies                OK
  gender                         OK


# One-Hot Encode Multi-Category Columns

In [4]:
# One-hot encode columns with 3+ categories
# WHY NOT a simple number map?
# Because that implies ORDER (0 < 1 < 2) — but contract types have no order
# One-hot creates a separate True/False column for each category value

multi_cat_columns = ['Contract', 'PaymentMethod', 'InternetService']
shape_before      = df.shape[1]

df = pd.get_dummies(df, columns=multi_cat_columns, drop_first=True)
# drop_first=True removes one redundant column per group
# If month-to-month=0 AND one-year=0, it MUST be two-year — no need for that column

new_cols = [c for c in df.columns if any(cat in c for cat in multi_cat_columns)]

print(f"One-hot encoding complete.")
print(f"Columns before: {shape_before}  →  Columns after: {df.shape[1]}")
print(f"\nNew dummy columns created ({len(new_cols)}):")
for col in new_cols:
    print(f"  {col}")
print(f"\nFull column list ({df.shape[1]} total):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

One-hot encoding complete.
Columns before: 20  →  Columns after: 24

New dummy columns created (7):
  Contract_One year
  Contract_Two year
  PaymentMethod_Credit card (automatic)
  PaymentMethod_Electronic check
  PaymentMethod_Mailed check
  InternetService_Fiber optic
  InternetService_No

Full column list (24 total):
   1. gender
   2. SeniorCitizen
   3. Partner
   4. Dependents
   5. tenure
   6. PhoneService
   7. MultipleLines
   8. OnlineSecurity
   9. OnlineBackup
  10. DeviceProtection
  11. TechSupport
  12. StreamingTV
  13. StreamingMovies
  14. PaperlessBilling
  15. MonthlyCharges
  16. TotalCharges
  17. Churn
  18. Contract_One year
  19. Contract_Two year
  20. PaymentMethod_Credit card (automatic)
  21. PaymentMethod_Electronic check
  22. PaymentMethod_Mailed check
  23. InternetService_Fiber optic
  24. InternetService_No


# Split Features and Target

In [6]:
# Separate inputs (X) from the answer we want to predict (y)
# X = "features" = everything the model uses to make its decision
# y = "target"   = Churn (0 = stayed, 1 = churned)

X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Feature matrix X:  {X.shape[0]:,} rows × {X.shape[1]} columns")
print(f"Target vector y:   {len(y):,} values")
print(f"\nTarget distribution:")
for label, count in y.value_counts().items():
    pct        = count / len(y) * 100
    label_name = 'Stayed  (0)' if label == 0 else 'Churned (1)'
    print(f"  {label_name}: {count:,} ({pct:.1f}%)")

os.makedirs('../models', exist_ok=True)
with open('../models/feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)
print(f"\nFeature column names saved → models/feature_columns.json")
print(f"Total features: {len(X.columns)}")

Feature matrix X:  7,032 rows × 23 columns
Target vector y:   7,032 values

Target distribution:
  Stayed  (0): 5,163 (73.4%)
  Churned (1): 1,869 (26.6%)

Feature column names saved → models/feature_columns.json
Total features: 23


# Train/Test Split

In [7]:
# Split data into training set (80%) and test set (20%)
# ANALOGY: Like a teacher giving practice problems (training)
# then a final exam with new problems the model has never seen (testing)
# stratify=y ensures the churn ratio is identical in both splits

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train/test split complete!")
print(f"\n{'Split':<12} {'Rows':>8} {'Churn %':>10}")
print(f"{'-'*32}")
print(f"{'Training':<12} {len(X_train):>8,} {y_train.mean()*100:>9.1f}%")
print(f"{'Test':<12} {len(X_test):>8,} {y_test.mean()*100:>9.1f}%")
print(f"{'Total':<12} {len(X):>8,} {y.mean()*100:>9.1f}%")
print(f"\nChurn rates are consistent across both splits — stratify=y is working correctly.")

Train/test split complete!

Split            Rows    Churn %
--------------------------------
Training        5,625      26.6%
Test            1,407      26.6%
Total           7,032      26.6%

Churn rates are consistent across both splits — stratify=y is working correctly.


# Scale Numerical Features

In [8]:
# StandardScaler — put all numbers on the same scale
# ANALOGY: Comparing heights in cm (150–200) vs bank balances (0–100,000)
# Without scaling the model pays too much attention to large numbers just
# because they're bigger — not because they're more important
# StandardScaler: (value - mean) / std_deviation → mean=0, std=1

scaler = StandardScaler()

# CRITICAL: fit_transform on TRAINING data only
# fit()       → scaler learns the mean and std FROM training data
# transform() → applies that scaling
X_train_scaled = scaler.fit_transform(X_train)

# CRITICAL: transform() ONLY on test data — NEVER fit() again
# Using the same scaling parameters ensures an honest evaluation
X_test_scaled = scaler.transform(X_test)

print(f"Scaling complete!")
print(f"\nTraining data statistics after scaling:")
print(f"  Mean (should be ~0.000): {X_train_scaled.mean():.4f}")
print(f"  Std  (should be ~1.000): {X_train_scaled.std():.4f}")
print(f"\nTest data statistics after scaling:")
print(f"  Mean: {X_test_scaled.mean():.4f}")
print(f"  Std:  {X_test_scaled.std():.4f}")

joblib.dump(scaler, '../models/scaler.pkl')
print(f"\nScaler saved → models/scaler.pkl")
print(f"IMPORTANT: This exact scaler must be used in the Streamlit app.")
print(f"           A different scaler = wrong scale = garbage predictions.")

Scaling complete!

Training data statistics after scaling:
  Mean (should be ~0.000): -0.0000
  Std  (should be ~1.000): 1.0000

Test data statistics after scaling:
  Mean: -0.0163
  Std:  0.9963

Scaler saved → models/scaler.pkl
IMPORTANT: This exact scaler must be used in the Streamlit app.
           A different scaler = wrong scale = garbage predictions.


# Handle Class Imbalance with SMOTE

In [9]:
# SMOTE — Synthetic Minority Over-sampling TEchnique
# PROBLEM:  74% of data is "No Churn" — a lazy model always predicts "stay"
#           and scores 74% accuracy but catches ZERO actual churners
# SOLUTION: Generate new realistic "Churn" examples by blending existing ones
# ANALOGY:  100 cat photos vs 1000 dog photos → SMOTE creates 900 new synthetic
#           cat photos so the model learns from equal examples on both sides
# CRITICAL RULE: SMOTE only on TRAINING data — NEVER on test data

print(f"Class balance BEFORE SMOTE:")
print(f"  Stayed  (0): {(y_train == 0).sum():,}  ({(y_train == 0).mean()*100:.1f}%)")
print(f"  Churned (1): {(y_train == 1).sum():,}  ({(y_train == 1).mean()*100:.1f}%)")
print(f"  Imbalance ratio: {(y_train == 0).sum() / (y_train == 1).sum():.1f}:1")

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"\nClass balance AFTER SMOTE:")
print(f"  Stayed  (0): {(y_train_res == 0).sum():,}  ({(y_train_res == 0).mean()*100:.1f}%)")
print(f"  Churned (1): {(y_train_res == 1).sum():,}  ({(y_train_res == 1).mean()*100:.1f}%)")
print(f"  Ratio now: 1:1 — perfectly balanced!")
print(f"  New training size: {len(X_train_res):,} rows (was {len(X_train):,})")

np.save('../models/X_train_res.npy', X_train_res)
np.save('../models/y_train_res.npy', y_train_res)
np.save('../models/X_test_scaled.npy', X_test_scaled)
np.save('../models/y_test.npy', y_test.values)
print(f"\nAll arrays saved to models/ folder.")

Class balance BEFORE SMOTE:
  Stayed  (0): 4,130  (73.4%)
  Churned (1): 1,495  (26.6%)
  Imbalance ratio: 2.8:1

Class balance AFTER SMOTE:
  Stayed  (0): 4,130  (50.0%)
  Churned (1): 4,130  (50.0%)
  Ratio now: 1:1 — perfectly balanced!
  New training size: 8,260 rows (was 5,625)

All arrays saved to models/ folder.
